# BetaTour: POPMUSIC'i tamamlayan 4+1 aday deneyi

Bu defter, mevcut BetaTour modelinin POPMUSIC aday grafiğinde **olmayan** yararlı kenarları seçip seçemediğini sınar. Her düğümün ilk dört adayı eşit-beşli POPMUSIC kontrolünden gelir; beşinci aday BetaTour veya öğrenmesiz tur sıklığı sıralamasından seçilir. Tüm adaylar beşli dosyada aynı sırayı ve LKH α-değerlerini kullanır.

**Çalıştırma:** Google Colab'da `Çalışma zamanı > Çalışma zamanı türünü değiştir` bölümünden A100 ya da L4 GPU ve High RAM seçin. En az 12 görünür vCPU gerekir. Ardından `Çalışma zamanı > Tümünü çalıştır` seçin ve Drive bağlantısına izin verin. Sonuçlar `MyDrive/BetaTour/residual40_v1/` altında birikir. Tamamlanınca ZIP Drive'a kaydedilir ve tarayıcıdan otomatik indirilir.

Colab kapanırsa aynı GPU ve vCPU türüyle yeniden `Tümünü çalıştır` seçin. Tamamlanmış örnekler atlanır. Farklı bir Colab sanal makinesinde yarım kalan **tek örnek** süre tutarlılığı için yeniden çalışır. Tahmini tam çalışma süresi 11–14 saattir; bu yalnızca yaklaşık bir planlama aralığıdır.

Bu ilk adım modeli yeniden eğitmez veya LKH'nin C kaynak kodunu değiştirmez. Arama geri bildirimiyle eğitim, ancak bu kontrollü deney yarar gösterirse ayrı bir deney olarak yapılmalıdır.


## 1. Sabit ayarlar ve analiz planı

Önceki 40 örnek kullanılmaz. Dört yeni grupta 40 sentetik örnek üretilir: uniform/kümeli 10.000 düğümden 12'şer ve 20.000 düğümden 8'er. 1.000 düğümlü örnekler önceki deneylerdeki yüksek beraberlik oranı nedeniyle bu deneyin kapsamı dışındadır. Üretim ve LKH tohumları aşağıdaki hücrede sabittir; sonuçlara bakıp değiştirmeyin.

**Birincil karşılaştırma:** BetaTour 4+1 / tur sıklığı 4+1. Örnek başına altı tohumun süre sınırındaki en iyi turu eşlenir; 40 örnekte iki yönlü kesin işaret testi (beraberlikler hariç, α=0,05) yapılır. Eşli ortalama yüzde fark için örnek düzeyinde bootstrap aralığı ayrıca verilir. İkincil karşılaştırmalar: kesintisiz POPMUSIC, yeniden başlatılmış POPMUSIC-5 ve sabit üç adaylı BetaTour. Üç ikincil p-değeri Holm ile düzeltilir. Alt gruplar yalnız betimleyicidir.


In [ ]:
# Colab: A100 or L4 GPU, High RAM and at least 12 visible vCPUs.
# Keep these settings fixed after the first instance has been saved.
EXPERIMENT_NAME = "residual40_v1"
PROFILE = "residual40"
GROUPS = ["uni10k", "clu10k", "uni20k", "clu20k"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 12, "uniform", 3103, 120.0],
    ["clu10k", 10000, 12, "clustered", 3204, 120.0],
    ["uni20k", 20000, 8, "uniform", 3305, 300.0],
    ["clu20k", 20000, 8, "clustered", 3406, 300.0],
]
MAX_NEW_INSTANCES = None  # e.g. 3 stops after three new instances this session
SAVE_TO_DRIVE = True
INCLUDE_ARRAYS_IN_ZIP = True  # phase1 and candidate NPZ files permit independent checks

W = 6
REQUIRE_12_VCPUS = True
SEEDS = list(range(201, 207))
K_GRAPH = 16
K_LIST = 5
K_LEARNED = 3  # fixed BetaTour comparator
RESIDUAL_POP_SLOTS = 4
PAD_SECONDS = 3.0
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
ARM_NAMES = ["pop_full", "restart_pop_5", "betatour_fixed3",
             "residual_frequency4p1", "residual_betatour4p1"]

assert W == len(SEEDS) and K_LEARNED == 3 and K_LIST == 5 and K_GRAPH == 16
assert RESIDUAL_POP_SLOTS == 4
assert PROFILE == "residual40" and sum(spec[2] for spec in INSTANCE_SPECS) == 40

ANALYSIS_PLAN = {
    "unit": "instance",
    "sample_size": 40,
    "endpoint": "lowest tour cost among six seeds by each instance's wall-clock deadline",
    "primary": "residual_betatour4p1 vs residual_frequency4p1",
    "primary_test": "two-sided exact sign test on non-tied instance pairs, alpha=0.05",
    "effect": "mean paired 100*(residual_betatour4p1-residual_frequency4p1)/residual_frequency4p1",
    "key_practical_comparison": "residual_betatour4p1 vs pop_full",
    "secondary": ["residual_betatour4p1 vs restart_pop_5",
                  "residual_betatour4p1 vs betatour_fixed3"],
    "multiplicity": "primary reported separately; Holm over three secondary overall comparisons",
    "subgroups": "10k/20k and uniform/clustered descriptive only",
    "post_result_tuning": "none; any 3+2 or alpha-order changes need new instances",
}


## 2. Ortam, model, Drive ve LKH

Sabit kaynak commit'i ile model SHA256 değeri doğrulanır. LKH-3.0.14 resmi kaynaktan, o erişilemezse SHA256 denetimli yedekten derlenir. Drive'daki `BetaTour` klasörü ve ayrı `residual40_v1` deney klasörü otomatik açılır. Manifest önce yazılır; yeniden çalıştırmada ayarların değişmediği denetlenir. GPU türü ve vCPU sayısı oturumlar arasında aynı kalmalıdır.


In [ ]:
import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tarfile
import time
import urllib.request
import uuid
from pathlib import Path

for package, module in [("scipy", "scipy"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import scipy
import torch

torch.set_num_threads(1)
assert torch.cuda.is_available(), "Colab'da GPU runtime seçin: L4 veya A100."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= W, f"Altı LKH süreci için en az {W} vCPU gerekli; görünen: {CPU_COUNT}"
if REQUIRE_12_VCPUS:
    assert CPU_COUNT >= 2 * W, f"Makaledeki 12 vCPU düzeni için 12 vCPU gerekli; görünen: {CPU_COUNT}"
if CPU_COUNT < 2 * W:
    print(f"UYARI: {CPU_COUNT} vCPU var. Bu çalışma makaledeki 12 vCPU düzenini tekrarlamıyor.")

def cpu_model_name():
    path = Path("/proc/cpuinfo")
    if path.exists():
        for line in path.read_text().splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

CPU_MODEL = cpu_model_name()

WORK_ROOT = Path("/content/betatour_validation_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == REPO_COMMIT
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M

checkpoint_path = REPO / "checkpoints" / "betatour_v2.pt"
actual_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
assert actual_hash == CHECKPOINT_SHA256, "Model dosyası beklenen SHA256 değeriyle eşleşmiyor."
MODEL, CK = M.load_model(str(checkpoint_path), DEVICE)  # pinned, trusted repository
assert MODEL.cfg.get("use_ensemble") is True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BETATOUR = Path("/content/drive/MyDrive/BetaTour")
    DRIVE_BETATOUR.mkdir(parents=True, exist_ok=True)
    OUTPUT_ROOT = DRIVE_BETATOUR / EXPERIMENT_NAME
else:
    OUTPUT_ROOT = WORK_ROOT / "results" / EXPERIMENT_NAME
for sub in ["instances", "stages", "candidates", "errors", "summary", "references"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

HARDWARE = {
    "gpu": GPU_NAME,
    "vcpus_visible": CPU_COUNT,
    "cpu_model": CPU_MODEL,
    "ram_gb": T.env_info().get("ram_gb"),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
}
MANIFEST = {
    "schema": 2,
    "experiment_name": EXPERIMENT_NAME,
    "profile": PROFILE,
    "groups": GROUPS,
    "instance_specs": INSTANCE_SPECS,
    "residual_rule": {
        "pop_slots": RESIDUAL_POP_SLOTS,
        "total_slots": K_LIST,
        "novelty": "undirected edge absent from seed's entire filled POP5 graph",
        "model_choice": "first novel edge in model rank",
        "frequency_choice": "first novel edge in tour-frequency rank",
        "position": "fifth; alpha defaults to 400 in candidate file",
        "fallback": "nearest novel edge among 64/128 neighbours, then all nodes if necessary",
    },
    "seeds": SEEDS,
    "workers": W,
    "graph_k": K_GRAPH,
    "candidate_k": K_LIST,
    "learned_k_for_fixed_comparator": K_LEARNED,
    "repo_commit": REPO_COMMIT,
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "hardware": HARDWARE,
    "arms": ARM_NAMES,
    "analysis_plan": ANALYSIS_PLAN,
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in ["schema", "profile", "groups", "instance_specs", "residual_rule", "seeds", "workers", "graph_k",
                "candidate_k", "learned_k_for_fixed_comparator", "repo_commit", "checkpoint_sha256", "arms", "analysis_plan"]:
        assert saved[key] == MANIFEST[key], f"{key} değişti. Ayrı EXPERIMENT_NAME kullanın."
    for key in ["gpu", "vcpus_visible"]:
        assert saved["hardware"][key] == HARDWARE[key], \
            f"Donanım {key} değişti. Eşli süre karşılaştırması için aynı Colab türünü seçin."
    if saved["hardware"] != HARDWARE:
        print("UYARI: CPU modeli, RAM veya kütüphane sürümleri değişmiş; her örneğin donanımı ayrıca kaydedilecek.")
else:
    manifest_tmp = manifest_path.with_name("manifest.json.tmp")
    manifest_tmp.write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
    os.replace(manifest_tmp, manifest_path)

# This local marker survives a notebook rerun on the same Colab VM. A fresh VM
# gets a new marker: incomplete instances are then rerun on one machine.
session_path = WORK_ROOT / "runtime_session_id.txt"
if session_path.exists():
    SESSION_ID = session_path.read_text().strip()
else:
    SESSION_ID = uuid.uuid4().hex
    session_path.write_text(SESSION_ID)

def get_lkh_3014():
    """Official source first; pinned SHA256 checked source mirror if it is unavailable."""
    root = WORK_ROOT / "lkh"
    binary = root / "LKH-3.0.14" / "LKH"
    if binary.exists():
        return str(binary)
    root.mkdir(parents=True, exist_ok=True)
    options = [
        ("resmi", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("SHA256 denetimli yedek", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar",
         "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33", 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout in options:
        archive = root / ("LKH-3.0.14.tar" if url.endswith(".tar") else "LKH-3.0.14.tgz")
        try:
            print("LKH kaynağı:", label, url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, archive.open("wb") as f:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    f.write(block)
            if expected_sha:
                got = hashlib.sha256(archive.read_bytes()).hexdigest()
                assert got == expected_sha, f"LKH yedek SHA256 uyuşmazlığı: {got}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(root, filter="data")
                except TypeError:
                    tar.extractall(root)
            assert (root / "LKH-3.0.14" / "Makefile").exists(), "LKH kaynak klasörü eksik"
            build = subprocess.run(["make", "-j", str(min(4, CPU_COUNT))],
                                   cwd=root / "LKH-3.0.14", stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, timeout=600)
            if build.returncode != 0:
                raise RuntimeError("LKH derlemesi başarısız:\n" + build.stdout[-3000:])
            assert binary.exists(), "LKH ikilisi oluşmadı"
            source_record = root / "build_source.json"
            source_tmp = root / "build_source.json.tmp"
            source_tmp.write_text(json.dumps({
                "source_label": label,
                "source_url": url,
                "source_archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),
                "binary_sha256": hashlib.sha256(binary.read_bytes()).hexdigest(),
            }, indent=2))
            os.replace(source_tmp, source_record)
            return str(binary)
        except Exception as exc:
            last_error = exc
            print(f"{label} başarısız: {exc}")
    raise RuntimeError(f"LKH 3.0.14 indirilemedi/derlenemedi: {last_error}")

LKH = get_lkh_3014()
build_source = WORK_ROOT / "lkh" / "build_source.json"
lkh_audit = json.loads(build_source.read_text()) if build_source.exists() else {}
lkh_audit.update({
    "binary_sha256": hashlib.sha256(Path(LKH).read_bytes()).hexdigest(),
    "session_id": SESSION_ID,
    "lkh_version": "3.0.14",
})
lkh_audit_path = OUTPUT_ROOT / f"lkh_build_{SESSION_ID}.json"
lkh_audit_tmp = lkh_audit_path.with_suffix(".json.tmp")
lkh_audit_tmp.write_text(json.dumps(lkh_audit, indent=2, ensure_ascii=False))
os.replace(lkh_audit_tmp, lkh_audit_path)
print("Hazır:", GPU_NAME, "|", CPU_COUNT, "vCPU | model", sum(p.numel() for p in MODEL.parameters()),
      "parametre | LKH", LKH, "| sonuçlar", OUTPUT_ROOT)


Mounted at /content/drive
LKH kaynağı: resmi https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz
resmi başarısız: <urlopen error timed out>
LKH kaynağı: SHA256 denetimli yedek https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar
Hazır: NVIDIA A100-SXM4-80GB | 12 vCPU | model 1875787 parametre | LKH /content/betatour_validation_work/lkh/LKH-3.0.14/LKH | sonuçlar /content/drive/MyDrive/BetaTour/residual40_v1


## 3. Yeni örnekler

Üretim tohumları 3103, 3204, 3305 ve 3406'dır; eski adaptif40 setinin 2101–2606 tohumları yeniden kullanılmaz. Bütçe 10.000 düğüm için 120 saniye, 20.000 düğüm için 300 saniyedir. Bu küme yalnız sentetik uniform ve kümeli dağılımları kapsar; TSPLIB veya özel olarak zorlaştırılmış örnekleri temsil etmez.


In [ ]:
def make_instances():
    specs = INSTANCE_SPECS
    result = []
    for group, n, count, dist, gen_seed, budget in specs:
        rng = np.random.default_rng(gen_seed)
        for i in range(count):
            xy = T.to_int_coords(T.gen_instance(n, dist, rng))
            result.append(dict(group=group, name=f"residual_{group}_{i:03d}", xy=xy,
                               ew="EUC_2D", problem_file=None, budget=budget, optimum=None))
    result.sort(key=lambda x: (x["budget"], x["name"]))
    assert len(result) == 40 and len({x["name"] for x in result}) == 40
    print(f"{len(result)} örnek hazır; tek kol bütçe toplamı {sum(x['budget'] for x in result)/3600:.2f} saat")
    return result

INSTANCES = make_instances()


40 örnek hazır; tek kol bütçe toplamı 2.13 saat


## 4. Eşlenik beş kollu deney

Kollar: kesintisiz POPMUSIC; aynı ilk turlardan yeniden başlayan eşit-beşli POPMUSIC; önceki sabit-üç BetaTour karışımı; tur sıklığı 4+1; BetaTour 4+1. Yeniden başlayan dört kol aynı altı ilk turu kullanır. İlk deneme maliyetleri kesintisiz kolla kontrol edilir. Her kolun ilk aşama ve kendine özgü hazırlık süresi kendi 120/300 saniyelik bütçesinden düşülür; Drive yazımı kıyas süresine girmez.

4+1 kollarının ilk dört girişi aynı `POPMUSIC-5` listesidir. Beşinci kenar, o tohumun POPMUSIC-5 aday grafiğinde **iki yönde de** bulunmayan ilk sıralı kenardır. Bazı ham POPMUSIC satırları dört adaydan azsa eksik yuvalar en yakın komşularla doldurulur ve oranı raporlanır. Her iki 4+1 kolunda yeni kenar beşinci konumdadır (α=400). Aday matrisleri ZIP'e eklenir.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import shutil
import traceback

POP_PARAMS = {"CANDIDATE_SET_TYPE": "POPMUSIC", "SUBGRADIENT": "NO", "MAX_CANDIDATES": K_LIST}

def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

def write_npz_atomic(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)

def parallel_tasks(func, kwargs):
    with ThreadPoolExecutor(max_workers=W) as pool:
        futures = [pool.submit(func, **kw) for kw in kwargs]
        return [future.result() for future in futures]

def instance_common(inst):
    return dict(lkh_bin=LKH, workroot=str(WORK_ROOT / "lkh_tmp"), xy=inst["xy"],
                problem_file=inst["problem_file"], ew=inst["ew"])

def event_list(res, offset):
    return [[int(cost), float(t)] for cost, t in T.abs_events(res, offset=offset)]

def best_within(events, budget):
    costs = [int(cost) for cost, t in events if t <= budget + 1e-9]
    return min(costs) if costs else None

def make_full_arm(inst):
    budget = inst["budget"]
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_anytime, [dict(common, budget=budget, seed=seed,
                                             params=POP_PARAMS, timeout=budget + PAD_SECONDS)
                                          for seed in SEEDS])
    runs = []
    for seed, res in zip(SEEDS, raw):
        assert res["returncode"] in (0, None) and (res["returncode"] == 0 or res["timed_out"]), (seed, res)
        events = event_list(res, res["t_start"] - start)
        if not events and res["cost_min"] is not None and res["wall"] <= budget:
            events = [[int(res["cost_min"]), float(res["t_start"] - start + res["wall"])]]
        best = best_within(events, budget)
        assert best is not None, f"Kesintisiz POPMUSIC: tohum {seed} için bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, best_at_budget=int(best),
                         wall_s=float(res["wall"]), returncode=res["returncode"],
                         timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                wall_s=float(time.time() - start))

def make_phase1(inst):
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_first_trial, [dict(common, seed=seed,
                                                  timeout=inst["budget"] + PAD_SECONDS)
                                               for seed in SEEDS])
    phase_wall = time.time() - start
    n = len(inst["xy"])
    tours, cands, costs, first_events = [], [], [], []
    for seed, res in zip(SEEDS, raw):
        assert T.is_valid_tour(res["tour"], n), f"İlk tur geçersiz: {seed}"
        assert res["cand"].shape == (n, K_LIST), f"Aday boyutu yanlış: {seed}"
        events = event_list(res, res["t_start"] - start)
        if not events:
            events = [[int(res["cost"]), float(res["t_start"] - start + res["wall"])]]
        assert best_within(events, inst["budget"]) is not None, f"İlk tur bütçe dışında: {seed}"
        tours.append(res["tour"])
        cands.append(res["cand"])
        costs.append(res["cost"])
        first_events.append(events)
    arrays = dict(tours=np.stack(tours).astype(np.int32),
                  pop_candidates=np.stack(cands).astype(np.int32),
                  costs=np.asarray(costs, dtype=np.float64))
    meta = dict(wall_s=float(phase_wall), first_events=first_events,
                first_costs=[float(c) for c in costs])
    return arrays, meta

def valid_candidates(cands, n):
    assert cands.shape == (W, n, K_LIST), cands.shape
    assert np.all((cands >= 0) & (cands < n)), "Eksik/geçersiz aday bulundu"
    assert not np.any(cands == np.arange(n)[None, :, None]), "Düğüm kendi adayı olamaz"
    for i in range(K_LIST):
        for j in range(i + 1, K_LIST):
            assert not np.any(cands[:, :, i] == cands[:, :, j]), "Tekrarlı aday bulundu"

def _undirected_keys(n, neighbours):
    """Encode valid (i,j) pairs so both directions have one key."""
    a = np.arange(n, dtype=np.int64)[:, None]
    b = np.asarray(neighbours, dtype=np.int64)
    assert b.ndim == 2 and b.shape[0] == n
    assert np.all((b >= 0) & (b < n)) and not np.any(a == b)
    return np.minimum(a, b) * np.int64(n) + np.maximum(a, b)


def choose_strict_novel(pop5, rank, xy):
    """Keep four equal-POP candidates; add one undirected POP-novel edge.

    `rank` has 16 unique graph neighbours per node, ordered by the arm's own
    method. If all 16 are in POP's *undirected* graph, widen only that arm's
    search to nearest 64/128 neighbours, then all nodes if necessary. The
    fallback is timed and reported.
    """
    n = len(xy)
    assert pop5.shape == (W, n, K_LIST)
    assert rank.shape == (n, K_GRAPH)
    rank_keys = _undirected_keys(n, rank)
    out = np.empty((W, n, K_LIST), dtype=np.int32)
    rank_position = np.zeros((W, n), dtype=np.int16)
    fallback = np.zeros((W, n), dtype=bool)
    rows = np.arange(n)
    for m in range(W):
        old = np.unique(_undirected_keys(n, pop5[m]))
        allowed = ~np.isin(rank_keys, old)
        has = allowed.any(axis=1)
        at = allowed.argmax(axis=1)
        picked = rank[rows, at].copy()
        rank_position[m, has] = (at[has] + 1).astype(np.int16)
        missing = ~has
        if missing.any():
            fallback[m, missing] = True
            for width in (64, 128):
                near, _ = T.knn(xy, width)
                available = ~np.isin(_undirected_keys(n, near), old)
                found = available.any(axis=1) & missing
                picked[found] = near[rows[found], available.argmax(axis=1)[found]]
                missing[found] = False
                if not missing.any():
                    break
            if missing.any():
                # Extremely rare high-degree POP node: scan all nodes by distance
                # only for the affected rows. This keeps the experiment resumable.
                xy64 = np.asarray(xy, dtype=np.float64)
                for i in np.flatnonzero(missing):
                    distance2 = ((xy64 - xy64[i]) ** 2).sum(axis=1)
                    order = np.argsort(distance2, kind="stable")
                    edge_keys = np.minimum(i, order) * np.int64(n) + np.maximum(i, order)
                    allowed_all = (order != i) & ~np.isin(edge_keys, old)
                    assert allowed_all.any(), "No POP-novel edge exists for a node"
                    picked[i] = order[np.argmax(allowed_all)]
                    missing[i] = False
        assert not np.isin(_undirected_keys(n, picked[:, None]), old).any()
        out[m, :, :RESIDUAL_POP_SLOTS] = pop5[m, :, :RESIDUAL_POP_SLOTS]
        out[m, :, RESIDUAL_POP_SLOTS] = picked
    valid_candidates(out, n)
    assert np.array_equal(out[:, :, :RESIDUAL_POP_SLOTS],
                          pop5[:, :, :RESIDUAL_POP_SLOTS])
    meta = dict(fallback_count=int(fallback.sum()),
                fallback_fraction=float(fallback.mean()),
                mean_graph_rank=float(rank_position[~fallback].mean()) if (~fallback).any() else None)
    return out, rank_position, fallback, meta


def make_candidate_lists(inst, phase):
    xy = inst["xy"]
    n = len(xy)
    tours, raw_pop, costs = phase["tours"], phase["pop_candidates"], phase["costs"]
    assert raw_pop.shape == (W, n, K_LIST)
    assert np.all((raw_pop == -1) | ((raw_pop >= 0) & (raw_pop < n)))
    assert not np.any(raw_pop == np.arange(n)[None, :, None])

    start = time.perf_counter()
    succ = np.stack([T.tour_succ_pred(tour)[0] for tour in tours], axis=1).astype(np.int32)
    successor_s = time.perf_counter() - start

    start = time.perf_counter()
    nearest, _ = T.knn(xy, K_GRAPH)
    pop5 = np.stack([T.mix_candidates(raw_pop[m], nearest, k_first=K_LIST,
                                      k=K_LIST, fill=nearest) for m in range(W)]).astype(np.int32)
    pop5_s = time.perf_counter() - start
    valid_candidates(pop5, n)

    start = time.perf_counter()
    prediction = M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=costs)
    prediction_s = time.perf_counter() - start
    rank = np.asarray(prediction["rank"], dtype=np.int32)
    assert rank.shape == (n, K_GRAPH)

    start = time.perf_counter()
    fixed = np.stack([T.mix_candidates(rank, raw_pop[m], k_first=K_LEARNED,
                                       k=K_LIST, fill=rank[:, K_LEARNED:])
                      for m in range(W)]).astype(np.int32)
    fixed_s = time.perf_counter() - start

    start = time.perf_counter()
    residual_beta, beta_rank_pos, beta_fallback, beta_meta = choose_strict_novel(pop5, rank, xy)
    residual_beta_s = time.perf_counter() - start

    start = time.perf_counter()
    xyf = np.asarray(xy, dtype=np.float64)
    span = float((xyf.max(0) - xyf.min(0)).max()) or 1.0
    xyn = (xyf - xyf.min(0)) / span
    count_nb, count_d = M.knn_np(xyn, K_GRAPH)
    extra = int(MODEL.cfg.get("graph_extra", 0))
    if extra:
        count_nb, count_d, _ = M.extend_graph(xyn, count_nb, succ, extra)
    assert np.array_equal(count_nb, prediction["NB"]), "Frequency and model graphs differ"
    freq = T.ensemble_frequency(count_nb, succ)
    freq_rank = T.frequency_candidates(count_nb, count_d, freq, k=K_GRAPH).astype(np.int32)
    frequency_rank_s = time.perf_counter() - start

    start = time.perf_counter()
    residual_freq, freq_rank_pos, freq_fallback, freq_meta = choose_strict_novel(pop5, freq_rank, xy)
    residual_freq_s = time.perf_counter() - start

    for arr in (fixed, residual_beta, residual_freq):
        valid_candidates(arr, n)
    assert np.array_equal(residual_beta[:, :, :4], residual_freq[:, :, :4])
    # The novel edge can occur in an initial tour even though it is absent
    # from POP's candidate graph. This is a redundancy diagnostic, not use in search.
    initial_edge_fraction = []
    for m in range(W):
        initial_keys = np.unique(_undirected_keys(n, succ[:, m, None]))
        selected_keys = _undirected_keys(n, residual_beta[m, :, 4, None]).ravel()
        initial_edge_fraction.append(float(np.isin(selected_keys, initial_keys).mean()))
    meta = dict(successor_build_s=float(successor_s), pop5_build_s=float(pop5_s),
                model_predict_s=float(prediction["t_total"]),
                fixed_list_s=float(fixed_s),
                residual_betatour_select_s=float(residual_beta_s),
                residual_frequency_rank_s=float(frequency_rank_s),
                residual_frequency_select_s=float(residual_freq_s),
                restart_pop_5_overhead_s=float(pop5_s),
                betatour_fixed3_overhead_s=float(successor_s + prediction_s + fixed_s),
                residual_betatour4p1_overhead_s=float(successor_s + pop5_s + prediction_s + residual_beta_s),
                residual_frequency4p1_overhead_s=float(successor_s + pop5_s + frequency_rank_s + residual_freq_s),
                raw_pop_candidates_per_node=float((raw_pop >= 0).sum(axis=2).mean()),
                raw_pop_below_four_fraction=float(((raw_pop >= 0).sum(axis=2) < 4).mean()),
                pop4_nearest_fill_fraction=float((~(
                    pop5[:, :, :4, None] == raw_pop[:, :, None, :]).any(-1)).mean()),
                fixed_top3_in_raw_pop5=float(np.mean([
                    (fixed[m, :, :K_LEARNED, None] == raw_pop[m, :, None, :]).any(-1).mean()
                    for m in range(W)])),
                beta_novel_graph_rank=beta_meta["mean_graph_rank"],
                beta_novel_fallback_count=beta_meta["fallback_count"],
                beta_novel_fallback_fraction=beta_meta["fallback_fraction"],
                frequency_novel_graph_rank=freq_meta["mean_graph_rank"],
                frequency_novel_fallback_count=freq_meta["fallback_count"],
                frequency_novel_fallback_fraction=freq_meta["fallback_fraction"],
                beta_selected_in_own_first_tour_fraction=float(np.mean(initial_edge_fraction)))
    arrays = dict(restart_pop_5=pop5, betatour_fixed3=fixed,
                  residual_betatour4p1=residual_beta, residual_frequency4p1=residual_freq,
                  model_rank=rank, frequency_rank=freq_rank,
                  beta_selected_graph_rank=beta_rank_pos,
                  frequency_selected_graph_rank=freq_rank_pos,
                  beta_fallback=beta_fallback, frequency_fallback=freq_fallback)
    return arrays, meta

def make_resumed_arm(inst, phase, phase_meta, candidate_lists, overhead_s, arm_name):
    budget = inst["budget"]
    prefix = float(phase_meta["wall_s"]) + float(overhead_s)
    remaining = budget - prefix
    if remaining <= 0.25:
        runs = []
        for m, seed in enumerate(SEEDS):
            first = phase_meta["first_events"][m]
            best = best_within(first, budget)
            assert best is not None, f"{arm_name} / {seed}: ilk tur bütçe dışında"
            runs.append(dict(seed=seed, events=first, first_events=first, continuation_events=[],
                             best_at_budget=int(best), wall_s=0.0, returncode=None, timed_out=False))
        return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                    continuation_wall_s=0.0, conceptual_prelude_s=float(prefix),
                    continuation_budget_s=max(0.0, float(remaining)), skipped_due_budget=True)
    common = instance_common(inst)
    start = time.time()
    raw = parallel_tasks(T.task_guided, [dict(common, budget=remaining, seed=SEEDS[m],
                                             cand=candidate_lists[m], init_tour=phase["tours"][m],
                                             timeout=remaining + PAD_SECONDS)
                                          for m in range(W)])
    runs = []
    for m, (seed, res) in enumerate(zip(SEEDS, raw)):
        assert (res["returncode"] in (0, None) and
                (res["returncode"] == 0 or res["timed_out"])), \
            f"{arm_name} / {seed}: LKH çıkış kodu {res['returncode']}"
        first = phase_meta["first_events"][m]
        resumed = event_list(res, prefix + res["t_start"] - start)
        if res["cost_check"] is not None and res["returncode"] == 0:
            resumed.append([int(res["cost_check"]),
                            float(prefix + res["t_start"] - start + res["wall"])])
        events = sorted(first + resumed, key=lambda e: e[1])
        best = best_within(events, budget)
        assert best is not None, f"{arm_name} / {seed}: bütçe içinde tur yok"
        runs.append(dict(seed=seed, events=events, first_events=first, continuation_events=resumed,
                         best_at_budget=int(best), wall_s=float(res["wall"]),
                         returncode=res["returncode"], timed_out=bool(res["timed_out"])))
    return dict(runs=runs, best_at_budget=min(r["best_at_budget"] for r in runs),
                continuation_wall_s=float(time.time() - start),
                conceptual_prelude_s=float(prefix), continuation_budget_s=float(remaining))

def load_phase(stage_dir):
    meta_path, array_path = stage_dir / "phase1.json", stage_dir / "phase1.npz"
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def load_lists(stage_dir, name):
    meta_path, array_path = stage_dir / "candidates.json", OUTPUT_ROOT / "candidates" / (name + ".npz")
    if meta_path.exists() and array_path.exists():
        with np.load(array_path, allow_pickle=False) as z:
            arrays = {k: z[k] for k in z.files}
        return arrays, json.loads(meta_path.read_text())
    return None

def run_one_instance(inst, ordinal):
    name = inst["name"]
    final_path = OUTPUT_ROOT / "instances" / (name + ".json")
    if final_path.exists():
        data = json.loads(final_path.read_text())
        assert data.get("complete") is True and set(data["arms"]) == set(ARM_NAMES)
        print("ATLA (tamam):", name, flush=True)
        return data
    stage_dir = OUTPUT_ROOT / "stages" / name
    session_marker = stage_dir / "runtime_session_id.txt"
    if stage_dir.exists() and any(stage_dir.iterdir()):
        old_id = session_marker.read_text().strip() if session_marker.exists() else None
        if old_id != SESSION_ID:
            print("Yeni Colab VM: yarım kalan örnek aynı donanımda baştan çalıştırılıyor:", name, flush=True)
            shutil.rmtree(stage_dir)
            (OUTPUT_ROOT / "candidates" / (name + ".npz")).unlink(missing_ok=True)
    stage_dir.mkdir(parents=True, exist_ok=True)
    session_marker.write_text(SESSION_ID)
    xy_sha = hashlib.sha256(np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest()
    full_path = stage_dir / "pop_full.json"
    full = json.loads(full_path.read_text()) if full_path.exists() else None
    controls = ["restart_pop_5", "betatour_fixed3", "residual_frequency4p1",
                "residual_betatour4p1"]
    rng = np.random.default_rng(10000 + ordinal)
    control_order = [str(x) for x in rng.permutation(controls)]

    def ensure_full():
        nonlocal full
        if full is None:
            print(name, "/ kesintisiz POPMUSIC", flush=True)
            full = make_full_arm(inst)
            write_json_atomic(full_path, full)

    def ensure_controls():
        phase_loaded = load_phase(stage_dir)
        if phase_loaded is None:
            print(name, "/ altı ortak ilk deneme", flush=True)
            phase, phase_meta = make_phase1(inst)
            write_npz_atomic(stage_dir / "phase1.npz", **phase)
            write_json_atomic(stage_dir / "phase1.json", phase_meta)
        else:
            phase, phase_meta = phase_loaded
        lists_loaded = load_lists(stage_dir, name)
        if lists_loaded is None:
            print(name, "/ aday listeleri", flush=True)
            lists, list_meta = make_candidate_lists(inst, phase)
            write_npz_atomic(OUTPUT_ROOT / "candidates" / (name + ".npz"), **lists)
            write_json_atomic(stage_dir / "candidates.json", list_meta)
        else:
            lists, list_meta = lists_loaded
        for arm in control_order:
            path = stage_dir / (arm + ".json")
            if path.exists():
                continue
            print(name, "/", arm, flush=True)
            result = make_resumed_arm(inst, phase, phase_meta, lists[arm],
                                      list_meta[arm + "_overhead_s"], arm)
            write_json_atomic(path, result)
        return phase_meta, list_meta

    if ordinal % 2 == 0:
        ensure_full()
        phase_meta, list_meta = ensure_controls()
    else:
        phase_meta, list_meta = ensure_controls()
        ensure_full()
    arms = {arm: json.loads((stage_dir / (arm + ".json")).read_text()) for arm in ARM_NAMES}
    first_match = [int(arms["pop_full"]["runs"][m]["events"][0][0]) ==
                   int(phase_meta["first_events"][m][0][0]) for m in range(W)]
    data = dict(schema=2, complete=True, name=name, group=inst["group"], n=len(inst["xy"]),
                budget_s=float(inst["budget"]), optimum=inst["optimum"], ew=inst["ew"],
                xy_sha256=xy_sha, seeds=SEEDS, control_order=control_order,
                full_first=bool(ordinal % 2 == 0), hardware=HARDWARE,
                phase1=phase_meta, overheads=list_meta,
                first_trial_matches_uninterrupted=first_match, arms=arms)
    write_json_atomic(final_path, data)
    b = {arm: arms[arm]["best_at_budget"] for arm in ARM_NAMES}
    print(f"TAMAM {name}: full={b['pop_full']}, restart5={b['restart_pop_5']}, "
          f"fixed3={b['betatour_fixed3']}, freq4+1={b['residual_frequency4p1']}, "
          f"BetaTour4+1={b['residual_betatour4p1']}", flush=True)
    return data


## 5. Çalıştırma ve kaldığı yerden sürdürme

Bu hücre her örnek ve aşamayı Drive'a atomik kaydeder. Tamamlanmış örnekler yeniden çalışmaz. Aynı Colab sanal makinesinde kesilen bir kol aşamasından devam eder; yeni sanal makinede yalnız yarım kalan örnek baştan alınır. Hata olursa `errors/` klasörüne ayrıntı yazılır. `MAX_NEW_INSTANCES` ile bir oturumda kaç yeni örnek çalışacağını sınırlayabilirsiniz; tam 40 tamamlanmadan çıkarımsal p-değerleri üretilmez.


In [ ]:
def warmup():
    xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
    succ = np.stack([np.roll(np.arange(100), -1)] * W, axis=1)
    M.predict_tourfold(MODEL, xy, K_GRAPH, DEVICE, ens_succ=succ, ens_cost=np.ones(W))
    first = T.task_first_trial(LKH, str(WORK_ROOT / "lkh_tmp"), SEEDS[0], xy,
                               timeout=30)
    assert T.is_valid_tour(first["tour"], 100) and first["cand"].shape == (100, K_LIST)
    print("GPU ve LKH öz sınaması tamam.")

warmup()

def save_progress(last_instance=None, state="running"):
    finished = sorted(p.stem for p in (OUTPUT_ROOT / "instances").glob("*.json"))
    write_json_atomic(OUTPUT_ROOT / "progress.json",
                      dict(state=state, completed=len(finished), total=len(INSTANCES),
                           finished=finished, last_instance=last_instance,
                           updated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())))

save_progress(state="running")
session_start = time.time()
newly_finished = 0
for ordinal, inst in enumerate(INSTANCES):
    final_file = OUTPUT_ROOT / "instances" / (inst["name"] + ".json")
    if final_file.exists():
        record = json.loads(final_file.read_text())
        assert record.get("complete") and record.get("xy_sha256") == hashlib.sha256(
            np.ascontiguousarray(inst["xy"]).tobytes()).hexdigest(), \
            f"{inst['name']}: eski kayıt mevcut kurulumla uyuşmuyor"
        print("ATLA (tamam):", inst["name"], flush=True)
        continue
    if MAX_NEW_INSTANCES is not None and newly_finished >= MAX_NEW_INSTANCES:
        print("Bu oturum için MAX_NEW_INSTANCES sınırına ulaşıldı.")
        break
    try:
        run_one_instance(inst, ordinal)
        newly_finished += 1
        save_progress(last_instance=inst["name"])
    except Exception as exc:
        failure = dict(instance=inst["name"], error=repr(exc), traceback=traceback.format_exc(),
                       utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + ".json"), failure)
        save_progress(last_instance=inst["name"], state="error")
        raise

done = len(list((OUTPUT_ROOT / "instances").glob("*.json")))
save_progress(state="complete" if done == len(INSTANCES) else "partial")
print(f"Bu oturumda yeni {newly_finished}, toplam {done}/{len(INSTANCES)} örnek tamamlandı. "
      f"Süre: {(time.time()-session_start)/3600:.2f} saat. Kayıt: {OUTPUT_ROOT}")


GPU ve LKH öz sınaması tamam.
residual_clu10k_000 / kesintisiz POPMUSIC
residual_clu10k_000 / altı ortak ilk deneme
residual_clu10k_000 / aday listeleri
residual_clu10k_000 / betatour_fixed3
residual_clu10k_000 / residual_frequency4p1
residual_clu10k_000 / residual_betatour4p1
residual_clu10k_000 / restart_pop_5
TAMAM residual_clu10k_000: full=24652417, restart5=24651671, fixed3=24651923, freq4+1=24654066, BetaTour4+1=24651716
residual_clu10k_001 / altı ortak ilk deneme
residual_clu10k_001 / aday listeleri
residual_clu10k_001 / residual_betatour4p1
residual_clu10k_001 / betatour_fixed3
residual_clu10k_001 / residual_frequency4p1
residual_clu10k_001 / restart_pop_5
residual_clu10k_001 / kesintisiz POPMUSIC
TAMAM residual_clu10k_001: full=30167283, restart5=30168998, fixed3=30168550, freq4+1=30173273, BetaTour4+1=30170277
residual_clu10k_002 / kesintisiz POPMUSIC
residual_clu10k_002 / altı ortak ilk deneme
residual_clu10k_002 / aday listeleri
residual_clu10k_002 / residual_frequency4p1
r

## 6. Önceden belirlenen analiz

`summary/per_instance.csv` örnek düzeyindeki, `per_seed.csv` tohum düzeyindeki ham sonuçları içerir. `candidate_diagnostics.csv` POPMUSIC dışı aday seçimini, fallback oranlarını ve hazırlık sürelerini gösterir. Dört karşılaştırmanın kazanım/beraberlik/kayıpları, eşli yüzde farkları ve tam 40 örnek olduğunda testleri `paired_stats.csv` ile `RAPOR.md` içine yazılır.

**Yorum:** Birincil test BetaTour'un aynı 4+1 kuralındaki basit tur sıklığına özel üstünlüğünü sınar. Kesintisiz POPMUSIC karşılaştırması gerçek uçtan uca yararı gösterir. İlk turlar eşleşmezse çıkarımsal sonuç üretilmez. Seçilen bir kenarın ilk turda bulunması yalnız gereksizlik göstergesidir; mevcut LKH çıktısı kenarın arama içinde denenme sayısını vermediği için böyle bir kullanım iddiası yapılmaz.


In [ ]:
from scipy.stats import binomtest
from IPython.display import display


def load_finished():
    by_name = {x["name"]: x for x in INSTANCES}
    records = []
    seen = set()
    for path in sorted((OUTPUT_ROOT / "instances").glob("*.json")):
        item = json.loads(path.read_text())
        if item.get("complete") and item["name"] in by_name:
            assert path.stem == item["name"], f"Result filename/name mismatch: {path}"
            assert item["name"] not in seen, f"Duplicate result: {item['name']}"
            seen.add(item["name"])
            assert set(item["arms"]) == set(ARM_NAMES)
            assert all(len(item["arms"][arm]["runs"]) == W for arm in ARM_NAMES)
            expected_xy = hashlib.sha256(np.ascontiguousarray(by_name[item["name"]]["xy"]).tobytes()).hexdigest()
            assert item["xy_sha256"] == expected_xy
            records.append(item)
    return records


def verify_saved_candidates(item):
    """Audit the exact candidate arrays preserved in Drive and in the ZIP."""
    n, name = int(item["n"]), item["name"]
    path = OUTPUT_ROOT / "candidates" / (name + ".npz")
    assert path.exists(), f"Missing candidate array: {path}"
    with np.load(path, allow_pickle=False) as z:
        pop5 = z["restart_pop_5"]
        beta = z["residual_betatour4p1"]
        frequency = z["residual_frequency4p1"]
        for arr in (pop5, beta, frequency, z["betatour_fixed3"]):
            valid_candidates(arr, n)
        assert np.array_equal(beta[:, :, :4], pop5[:, :, :4])
        assert np.array_equal(frequency[:, :, :4], pop5[:, :, :4])
        for m in range(W):
            old = np.unique(_undirected_keys(n, pop5[m]))
            for arr in (beta, frequency):
                chosen = _undirected_keys(n, arr[m, :, 4, None]).ravel()
                assert not np.isin(chosen, old).any(), f"Non-novel fifth edge: {name}, seed {SEEDS[m]}"
    return True


records = load_finished()
for item in records:
    verify_saved_candidates(item)
mismatch = [r["name"] for r in records if not all(r["first_trial_matches_uninterrupted"])]
rows, seed_rows, diagnostic_rows = [], [], []
COMPARISONS = [
    ("BetaTour 4+1 / frequency 4+1 [primary]", "residual_betatour4p1", "residual_frequency4p1", "primary"),
    ("BetaTour 4+1 / continuous POP [practical]", "residual_betatour4p1", "pop_full", "secondary"),
    ("BetaTour 4+1 / restarted POP5", "residual_betatour4p1", "restart_pop_5", "secondary"),
    ("BetaTour 4+1 / fixed BetaTour3", "residual_betatour4p1", "betatour_fixed3", "secondary"),
]
for item in records:
    b = float(item["budget_s"])
    row = dict(instance=item["name"], group=item["group"], n=item["n"], budget_s=b,
               first_match_all=all(item["first_trial_matches_uninterrupted"]),
               phase1_s=item["phase1"]["wall_s"], cpu_model=item["hardware"]["cpu_model"],
               gpu=item["hardware"]["gpu"])
    for arm in ARM_NAMES:
        arm_data = item["arms"][arm]
        row[arm] = int(arm_data["best_at_budget"])
        for run in arm_data["runs"]:
            seed_rows.append(dict(instance=item["name"], group=item["group"], n=item["n"],
                                  seed=run["seed"], arm=arm, cost=run["best_at_budget"], budget_s=b))
    for label, treatment, control, kind in COMPARISONS:
        row[treatment + "_vs_" + control + "_abs"] = row[treatment] - row[control]
        row[treatment + "_vs_" + control + "_pct"] = 100.0 * (row[treatment] - row[control]) / row[control]
    rows.append(row)
    diag = dict(instance=item["name"], group=item["group"], n=item["n"],
                phase1_s=item["phase1"]["wall_s"])
    diag.update(item["overheads"])
    diagnostic_rows.append(diag)

per_instance = pd.DataFrame(rows)
per_seed = pd.DataFrame(seed_rows)
diagnostics = pd.DataFrame(diagnostic_rows)
per_instance.to_csv(OUTPUT_ROOT / "summary" / "per_instance.csv", index=False)
per_seed.to_csv(OUTPUT_ROOT / "summary" / "per_seed.csv", index=False)
diagnostics.to_csv(OUTPUT_ROOT / "summary" / "candidate_diagnostics.csv", index=False)


def paired_stats(values, bootstrap_seed, inferential):
    x = np.asarray(values, dtype=float)
    assert np.isfinite(x).all() and len(x)
    wins, ties, losses = int((x < 0).sum()), int((x == 0).sum()), int((x > 0).sum())
    if inferential:
        p = float(binomtest(wins, wins + losses, 0.5).pvalue) if wins + losses else 1.0
        rng = np.random.default_rng(bootstrap_seed)
        means = x[rng.integers(0, len(x), size=(5000, len(x)))].mean(axis=1)
        lo, hi = (float(v) for v in np.quantile(means, [0.025, 0.975]))
    else:
        p, lo, hi = np.nan, np.nan, np.nan
    return dict(instances=len(x), win=wins, tie=ties, loss=losses,
                mean_delta_pct=float(x.mean()), median_delta_pct=float(np.median(x)),
                ci95_low_pct=lo, ci95_high_pct=hi, sign_p=p)


inference_ready = len(records) == len(INSTANCES) and len(INSTANCES) == 40 and not mismatch
stats_rows = []
if len(per_instance):
    groups = [("all", per_instance)] + list(per_instance.groupby("group", sort=True))
    for group, frame in groups:
        for i, (label, treatment, control, kind) in enumerate(COMPARISONS):
            column = treatment + "_vs_" + control + "_pct"
            stats_rows.append(dict(group=group, comparison=label, class_of_comparison=kind,
                                   **paired_stats(frame[column].to_numpy(), 20261026 + i,
                                                  inferential=(group == "all" and inference_ready))))
stats = pd.DataFrame(stats_rows)
if len(stats):
    stats["sign_p_holm_secondary"] = np.nan
    secondary_ix = stats.index[(stats["group"] == "all") & (stats["class_of_comparison"] == "secondary")].tolist()
    if inference_ready:
        ordered = sorted(secondary_ix, key=lambda j: stats.loc[j, "sign_p"])
        running = 0.0
        for rank, j in enumerate(ordered):
            running = max(running, (len(ordered) - rank) * float(stats.loc[j, "sign_p"]))
            stats.loc[j, "sign_p_holm_secondary"] = min(1.0, running)
stats.to_csv(OUTPUT_ROOT / "summary" / "paired_stats.csv", index=False)

if mismatch:
    note = (f"INVALID COMPARISON: first-trial tour costs disagree on {len(mismatch)} instances. "
            "Raw data are retained, but inferential p-values are withheld.")
elif not inference_ready:
    note = f"PARTIAL RESULT: {len(records)}/40 instances complete; no inferential p-values yet."
else:
    note = "Complete: 40/40 new 10k/20k synthetic instances. Primary: BetaTour 4+1 vs frequency 4+1."

cpu_models = sorted({r["hardware"]["cpu_model"] for r in records})
lines = ["# BetaTour residual 4+1 candidate experiment", "", note, "",
         "Each 4+1 list retains four equal-five POPMUSIC entries and appends one undirected POP-novel edge.",
         "The learned and frequency arms use the same four POP entries and novel-edge position (fifth, alpha=400).",
         "Percent difference = 100 * (first arm - second arm) / second arm; negative favours the first arm.",
         "Primary: BetaTour 4+1 versus frequency 4+1, exact two-sided sign test on 40 instance pairs, ties omitted.",
         "The three secondary overall sign tests receive Holm correction; subgroup rows are descriptive only.",
         "Bootstrap intervals describe the mean paired percent difference and are not equivalent to sign-test p-values.",
         "Practical superiority requires comparing against continuous POPMUSIC at the same total deadline.",
         "The six common first tours, seed and instance are matched across restarted arms. First-trial costs are checked against continuous POP.",
         "Common first phase plus arm-specific preparation and continuation count against each deadline. Drive writes and archive export do not.",
         "Novel edge presence in an initial tour is a redundancy diagnostic, not proof of use during LKH search.",
         "This holdout covers synthetic uniform and clustered 10k/20k instances; it does not establish effects on 1k, TSPLIB or evolved hard instances.",
         "Incomplete data or first-trial mismatches suppress inferential statistics.", ""]
if len(cpu_models) > 1:
    lines += ["WARNING: instances used multiple CPU models: " + ", ".join(cpu_models), ""]
if mismatch:
    lines += ["First-trial mismatches: " + ", ".join(mismatch), ""]
if len(stats):
    lines += ["```", stats.round(6).to_string(index=False), "```", ""]
lines += ["Raw files: summary/per_instance.csv, summary/per_seed.csv, summary/candidate_diagnostics.csv.",
          "Default ZIP includes stages/phase1.npz and candidates/*.npz for candidate-list auditing."]
(OUTPUT_ROOT / "summary" / "RAPOR.md").write_text("\n".join(lines), encoding="utf-8")
print(note)
if mismatch:
    print("First-trial mismatches:", mismatch)
if len(stats):
    display(stats[stats["group"] == "all"].round(6))


Complete: 40/40 new 10k/20k synthetic instances. Primary: BetaTour 4+1 vs frequency 4+1.


,group,comparison,class_of_comparison,instances,win,tie,loss,mean_delta_pct,median_delta_pct,ci95_low_pct,ci95_high_pct,sign_p,sign_p_holm_secondary
0,all,BetaTour 4+1 / frequency 4+1 [primary],primary,40,33,0,7,-0.006204,-0.005078,-0.008955,-0.003571,0.000042,NaN
1,all,BetaTour 4+1 / continuous POP [practical],secondary,40,8,0,32,0.007127,0.006579,0.004216,0.010026,0.000182,0.000546
2,all,BetaTour 4+1 / restarted POP5,secondary,40,21,0,19,-0.002278,-0.001548,-0.006502,0.001663,0.874629,0.874629
3,all,BetaTour 4+1 / fixed BetaTour3,secondary,40,15,0,25,0.003221,0.002978,0.000093,0.006263,0.153860,0.307720


## 7. Tam ZIP ve otomatik indirme

ZIP; manifesti, bütün örnek JSON kayıtlarını, analiz CSV ve raporunu, `stages/phase1.npz` ilk tur/POPMUSIC dizilerini ve `candidates/*.npz` gerçek aday matrislerini içerir. Bu dizilerle 4+1 yenilik kuralı bağımsız olarak denetlenebilir. ZIP her çalışmada Drive'a kaydedilir; **40/40 tamamlandığında** tarayıcı indirmesi otomatik başlar. Kısmi ZIP ara yedek olarak yalnız Drive'a yazılır.


In [ ]:
import shutil
import zipfile

zip_name = EXPERIMENT_NAME + "_results.zip"
local_zip = WORK_ROOT / zip_name
with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    included = 0
    for path in OUTPUT_ROOT.rglob("*"):
        if not path.is_file() or path.suffix == ".tmp" or path.name.endswith(".tmp.npz"):
            continue
        if path.suffix == ".zip":
            continue
        relative = path.relative_to(OUTPUT_ROOT)
        if not INCLUDE_ARRAYS_IN_ZIP and path.suffix == ".npz":
            continue
        z.write(path, str(relative))
        included += 1

drive_zip = OUTPUT_ROOT / zip_name
drive_tmp = OUTPUT_ROOT / (zip_name + ".tmp")
shutil.copy2(local_zip, drive_tmp)
os.replace(drive_tmp, drive_zip)
print(f"ZIP hazır: {drive_zip} ({local_zip.stat().st_size / 1024**2:.1f} MB, {included} dosya)")
progress = json.loads((OUTPUT_ROOT / "progress.json").read_text())
if SAVE_TO_DRIVE and progress["completed"] == progress["total"] and progress["state"] == "complete":
    from google.colab import files
    files.download(str(local_zip))  # browser download starts automatically
elif SAVE_TO_DRIVE:
    print(f"Ara ZIP Drive'da kaydedildi ({progress['completed']}/{progress['total']}). "
          "Tamamlanınca Tümünü çalıştırın; son ZIP otomatik inecek.")
else:
    print("Drive kapalı; yerel ZIP:", local_zip)
